In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [25]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)

DATA_PATH = "/kaggle/input/datasets/sourish05/messy-fraud-dataset/messy_fraud_data.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

print("SHAPE")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print(f"Memory  : {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

print("\nCOLUMN NAMES")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

print("\nDTYPES")
print(df.dtypes)

SHAPE
Rows    : 505,000
Columns : 23
Memory  : 403.15 MB

COLUMN NAMES
 1. transaction_id
 2. customer_id
 3. timestamp
 4. amount
 5. merchant
 6. merchant_category
 7. card_type
 8. card_last4
 9. country
10. city
11. device_type
12. ip_address
13. is_international
14. txn_count_10min
15. txn_count_24h
16. avg_amount_30d
17. account_age_days
18. credit_limit
19. hour_of_day
20. is_night
21. distance_from_home_km
22. channel
23. fraud_label

DTYPES
transaction_id            object
customer_id               object
timestamp                 object
amount                    object
merchant                  object
merchant_category         object
card_type                 object
card_last4               float64
country                   object
city                      object
device_type               object
ip_address                object
is_international           int64
txn_count_10min            int64
txn_count_24h              int64
avg_amount_30d            object
account_age_days  

In [26]:
summary = pd.DataFrame({
    "dtype":        df.dtypes.astype(str),
    "non_null":     df.notna().sum(),
    "null_count":   df.isna().sum(),
    "null_pct":     (df.isna().mean() * 100).round(2),
    "unique":       df.nunique(dropna=True),
    "sample_value": [df[c].dropna().iloc[0] if df[c].notna().any() else None
                     for c in df.columns],
})

print("PER-COLUMN QUALITY TABLE")
print(summary.to_string())

PER-COLUMN QUALITY TABLE
                         dtype  non_null  null_count  null_pct  unique         sample_value
transaction_id          object    502459        2541      0.50  497204         TXN000104241
customer_id             object    499949        5051      1.00   52923           CUST034785
timestamp               object    504931          69      0.01  393809  2023-06-01 23:25:00
amount                  object    479789       25211      4.99   44969                 1.68
merchant                object    464599       40401      8.00     170               Zomato
merchant_category       object    505000           0      0.00      13               Travel
card_type               object    479761       25239      5.00      26           MASTERCARD
card_last4             float64    494889       10111      2.00    8999             8,711.00
country                 object    474682       30318      6.00      30                   KP
city                    object    403867      101133   

In [27]:
print("DUPLICATES")
print(f"Exact duplicate rows     : {df.duplicated().sum():,}")
if "transaction_id" in df.columns:
    print(f"Duplicate transaction_id : {df['transaction_id'].duplicated().sum():,}")

print("\nTARGET DISTRIBUTION")
if "fraud_label" in df.columns:
    counts = df["fraud_label"].value_counts(dropna=False)
    pct    = (df["fraud_label"].value_counts(normalize=True, dropna=False) * 100).round(3)
    print(pd.DataFrame({"count": counts, "pct": pct}))
    ratio = counts.get(0, 0) / max(counts.get(1, 1), 1)
    print(f"\nClass imbalance ratio (normal : fraud) = {ratio:.1f} : 1")

print("\nCARDINALITY CHECK — low-cardinality categoricals")
cat_cols = df.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    nun = df[col].nunique(dropna=True)
    if nun <= 30:
        print(f"\n{col} ({nun} unique values):")
        print(df[col].value_counts(dropna=False).head(15))

DUPLICATES
Exact duplicate rows     : 4,181
Duplicate transaction_id : 7,795

TARGET DISTRIBUTION
              count   pct
fraud_label              
0            479737 95.00
1             25263  5.00

Class imbalance ratio (normal : fraud) = 19.0 : 1

CARDINALITY CHECK — low-cardinality categoricals

merchant_category (13 unique values):
merchant_category
Wire Transfer    40723
Crypto           40643
Electronics      40631
Adult            39480
Gift Cards       39405
Luxury           39242
Gambling         38983
Travel           38301
Retail           37674
Entertainment    37611
Groceries        37523
Fuel             37488
Food             37296
Name: count, dtype: int64

card_type (26 unique values):
card_type
visa                  36974
VISA                  36944
Discover              36895
Diners Club           36817
mastercard            36785
 mastercard           36763
Visa                  36762
MasterCard            36725
AMERICAN EXPRESS      36721
AMEX                  

In [28]:
# FIX: amount is object dtype (has "123 USD" strings).
# Coerce to numeric locally for inspection WITHOUT mutating df.

def to_numeric_local(series):
    return pd.to_numeric(
        series.astype(str).str.replace(r"[^\d\.\-]", "", regex=True),
        errors="coerce"
    )

amount_num = to_numeric_local(df["amount"])

print("NUMERIC SUMMARY (amount coerced locally)")
num_cols = df.select_dtypes(include="number").columns.tolist()
if num_cols:
    print(df[num_cols].describe().T[["min", "25%", "50%", "75%", "max", "mean"]])

print("\nRANGE SANITY CHECKS (domain-invalid value counts)")
checks = {
    "amount < 0":                (amount_num < 0),
    "amount == 0":               (amount_num == 0),
    "amount > 1,000,000":        (amount_num > 1_000_000),
    "account_age_days < 0":      (pd.to_numeric(df["account_age_days"], errors="coerce") < 0),
    "credit_limit <= 0":         (pd.to_numeric(df["credit_limit"], errors="coerce") <= 0),
    "distance_from_home_km < 0": (pd.to_numeric(df["distance_from_home_km"], errors="coerce") < 0),
    "hour_of_day not in 0-23":   (~pd.to_numeric(df["hour_of_day"], errors="coerce").between(0, 23)),
    "txn_count_10min < 0":       (pd.to_numeric(df["txn_count_10min"], errors="coerce") < 0),
    "txn_count_24h < 0":         (pd.to_numeric(df["txn_count_24h"], errors="coerce") < 0),
}
for name, mask in checks.items():
    print(f"{name:35s}: {int(mask.fillna(False).sum()):,}")

print("\nOBJECT COLUMNS — whitespace + mixed-case scan")
obj_cols = df.select_dtypes(include="object").columns.tolist()
print(f"{'column':25s} {'whitespace':>12s} {'mixed_case':>12s}")
for col in obj_cols:
    s = df[col].dropna().astype(str)
    has_ws    = s.str.contains(r"^\s|\s$", regex=True).sum()
    has_mixed = s.apply(lambda x: x != x.lower() and x != x.upper()).sum()
    print(f"{col:25s} {has_ws:>12,} {has_mixed:>12,}")

print("\nAMOUNT COLUMN — string-typed entries")
if df["amount"].dtype == "object":
    non_numeric = amount_num.isna() & df["amount"].notna()
    print(f"Non-numeric amount entries   : {int(non_numeric.sum()):,}")
    print("Samples:", df.loc[non_numeric, "amount"].head(5).tolist())

print("\nTIMESTAMP COLUMN — mixed types")
if "timestamp" in df.columns:
    types = df["timestamp"].dropna().apply(lambda x: type(x).__name__).value_counts()
    print(types)

NUMERIC SUMMARY (amount coerced locally)
                              min        25%        50%        75%           max       mean
card_last4               1,000.00   3,256.00   5,505.00   7,747.00      9,998.00   5,503.01
is_international             0.00       0.00       0.00       0.00          1.00       0.18
txn_count_10min            -11.00       0.00       1.00       2.00         15.00       1.20
txn_count_24h              -10.00       2.00       3.00       4.00         21.00       3.24
account_age_days        -4,997.00   1,041.00   2,361.00   3,682.00  9,913,976.00   5,476.78
credit_limit          -100,000.00 100,000.00 200,000.00 500,000.00 99,999,999.00 409,143.01
hour_of_day                 -1.00       5.00      11.00      18.00        999.00      11.87
is_night                     0.00       0.00       0.00       0.00          1.00       0.21
distance_from_home_km   -1,522.60       6.06      14.89      30.93 49,974,131.00  26,311.44
fraud_label                  0.00      

In [30]:
# View the full table in Colab/Kaggle's interactive data grid
# In Kaggle: use the "Data" tab on the right; in Colab: use display() or Data Wrangler.

from IPython.display import display, HTML

# Full table (interactive pager) — works in Kaggle and Colab
display(df)

# If the table is too big, show a smaller window:
df.head(200)   # first 200 rows
# df.sample(50)  # random 50 rows

# Quick text preview of first 10 rows, all columns
print("FIRST 10 ROWS — ALL COLUMNS")
print(df.head(10).to_string())

# Last 10 rows
print("\nLAST 10 ROWS — ALL COLUMNS")
print(df.tail(10).to_string())

# Random 10 rows (better than head for spot-checking messiness)
print("\nRANDOM 10 ROWS — ALL COLUMNS")
print(df.sample(10, random_state=42).to_string())

,transaction_id,customer_id,timestamp,amount,merchant,merchant_category,card_type,card_last4,country,city,device_type,ip_address,is_international,txn_count_10min,txn_count_24h,avg_amount_30d,account_age_days,credit_limit,hour_of_day,is_night,distance_from_home_km,channel,fraud_label
0,TXN000104241,CUST034785,2023-06-01 23:25:00,1.68,Zomato,Travel,MASTERCARD,"8,711.00",KP,Moscow,web,NaN,1,1,4,35.68,"1,887.00","500,000.00",23.00,0,34.69,Online,0
1,TXN000199676,CUST029737,2023-10-19 21:12:00,36.3,PayPal,Groceries,Visa,"2,558.00",US,Berlin,Mobile,9.132.47.196,0,0,5,9.66,"4,040.00","1,000,000.00",21.00,0,16.23,pos,0
2,TXN000140199,CUST006649,2023-07-24 11:53:00,208.06,BestBuy,Fuel,visa,"7,896.00",DE,Paris,NaN,13.42.180.254,0,0,2,102.92,"4,183.00","1,000,000.00",11.00,0,75.44,ATM,0
3,TXN000132814,CUST022654,2023-07-13 18:11:00,81.1,NaN,Gambling,Visa,"5,692.00",FR,London,Tablet,175.180.134.50,0,1,3,16.72,"3,040.00","1,000,000.00",18.00,0,5.98,pos,0
4,TXN000408697,CUST017646,2024-08-20 01:09:00,12.38,Airbnb,Gift Cards,NaN,"3,965.00",CN,Lagos,NaN,131.152.190.3,0,2,4,446.07,902.00,"1,000,000.00",1.00,1,2.69,POS,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
504995,TXN000353004,CUST006216,2024-05-30 23:04:00,46.51,Delta Air,Luxury,Visa,"2,621.00",PK,Moscow,ios,NaN,0,1,3,9.12,"2,914.00","500,000.00",23.00,0,17.77,Cheque,0
504996,TXN000162926,CUST000826,2023-08-26 19:20:00,12.58,Binance,Wire Transfer,mastercard,"3,115.00",DE,LA,NaN,4.218.74.90,0,4,3,72.39,"2,577.00","500,000.00",19.00,0,2.98,Online,0
504997,TXN000334523,CUST048108,2024-05-03 23:53:00,54.09,Walmart,Food,AMEX,"3,015.00",CA,Delhi,android,NaN,0,0,7,11.99,"2,322.00","1,000,000.00",23.00,0,7.51,App,0
504998,TXN000363030,CUST048393,2024-06-14 14:59:00,208.63,Starbucks,Adult,Visa,"9,800.00",CN,NYC,Desktop,68.178.192.100,0,1,2,8.03,"1,302.00","200,000.00",14.00,0,15.96,Contactless,0


FIRST 10 ROWS — ALL COLUMNS
  transaction_id customer_id            timestamp  amount   merchant merchant_category    card_type  card_last4 country    city device_type       ip_address  is_international  txn_count_10min  txn_count_24h avg_amount_30d  account_age_days  credit_limit  hour_of_day  is_night  distance_from_home_km      channel  fraud_label
0   TXN000104241  CUST034785  2023-06-01 23:25:00    1.68     Zomato            Travel   MASTERCARD    8,711.00      KP  Moscow         web              NaN                 1                1              4          35.68          1,887.00    500,000.00        23.00         0                  34.69       Online            0
1   TXN000199676  CUST029737  2023-10-19 21:12:00    36.3     PayPal         Groceries         Visa    2,558.00      US  Berlin      Mobile     9.132.47.196                 0                0              5           9.66          4,040.00  1,000,000.00        21.00         0                  16.23          pos        

In [1]:
# Cell 1: Imports and global configuration
import os
import re
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = "/kaggle/input/datasets/sourish05/messy-fraud-dataset/messy_fraud_data.csv"
OUTPUT_DIR = "/kaggle/working"
CLEAN_PATH = os.path.join(OUTPUT_DIR, "cleaned_fraud_data.csv")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Configuration loaded.")
print(f"Input : {DATA_PATH}")
print(f"Output: {CLEAN_PATH}")

Configuration loaded.
Input : /kaggle/input/datasets/sourish05/messy-fraud-dataset/messy_fraud_data.csv
Output: /kaggle/working/cleaned_fraud_data.csv


In [2]:
# Cell 2: Load raw dataset and inspect initial state
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Shape           : {df.shape}")
print(f"Memory usage    : {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print(f"Columns         : {len(df.columns)}")
print(f"Fraud rate      : {df['fraud_label'].mean():.4f}")
print(f"Fraud count     : {int(df['fraud_label'].sum()):,}")
print("\nColumn dtypes:")
print(df.dtypes)
print("\nFirst 5 rows:")
df.head()

Shape           : (505000, 23)
Memory usage    : 403.15 MB
Columns         : 23
Fraud rate      : 0.0500
Fraud count     : 25,263

Column dtypes:
transaction_id            object
customer_id               object
timestamp                 object
amount                    object
merchant                  object
merchant_category         object
card_type                 object
card_last4               float64
country                   object
city                      object
device_type               object
ip_address                object
is_international           int64
txn_count_10min            int64
txn_count_24h              int64
avg_amount_30d            object
account_age_days         float64
credit_limit             float64
hour_of_day              float64
is_night                   int64
distance_from_home_km    float64
channel                   object
fraud_label                int64
dtype: object

First 5 rows:


,transaction_id,customer_id,timestamp,amount,merchant,merchant_category,card_type,card_last4,country,city,device_type,ip_address,is_international,txn_count_10min,txn_count_24h,avg_amount_30d,account_age_days,credit_limit,hour_of_day,is_night,distance_from_home_km,channel,fraud_label
0,TXN000104241,CUST034785,2023-06-01 23:25:00,1.68,Zomato,Travel,MASTERCARD,"8,711.00",KP,Moscow,web,NaN,1,1,4,35.68,"1,887.00","500,000.00",23.00,0,34.69,Online,0
1,TXN000199676,CUST029737,2023-10-19 21:12:00,36.3,PayPal,Groceries,Visa,"2,558.00",US,Berlin,Mobile,9.132.47.196,0,0,5,9.66,"4,040.00","1,000,000.00",21.00,0,16.23,pos,0
2,TXN000140199,CUST006649,2023-07-24 11:53:00,208.06,BestBuy,Fuel,visa,"7,896.00",DE,Paris,NaN,13.42.180.254,0,0,2,102.92,"4,183.00","1,000,000.00",11.00,0,75.44,ATM,0
3,TXN000132814,CUST022654,2023-07-13 18:11:00,81.1,NaN,Gambling,Visa,"5,692.00",FR,London,Tablet,175.180.134.50,0,1,3,16.72,"3,040.00","1,000,000.00",18.00,0,5.98,pos,0
4,TXN000408697,CUST017646,2024-08-20 01:09:00,12.38,Airbnb,Gift Cards,NaN,"3,965.00",CN,Lagos,NaN,131.152.190.3,0,2,4,446.07,902.00,"1,000,000.00",1.00,1,2.69,POS,0


In [3]:
# Cell 3: Capture baseline quality metrics before cleaning
baseline = {
    "rows": len(df),
    "columns": len(df.columns),
    "missing_cells": int(df.isna().sum().sum()),
    "missing_pct": round(df.isna().sum().sum() / df.size * 100, 2),
    "duplicate_rows": int(df.duplicated().sum()),
    "duplicate_txn_id": int(df["transaction_id"].duplicated().sum()),
    "fraud_rate": round(df["fraud_label"].mean(), 4),
}

baseline_df = pd.DataFrame([baseline]).T.rename(columns={0: "value"})
print("BASELINE QUALITY SNAPSHOT")
print(baseline_df)

BASELINE QUALITY SNAPSHOT
                      value
rows             505,000.00
columns               23.00
missing_cells    602,339.00
missing_pct            5.19
duplicate_rows     4,181.00
duplicate_txn_id   7,795.00
fraud_rate             0.05


In [4]:
# Cell 4: Remove rows/columns that are entirely null
before_shape = df.shape

df.dropna(how="all", inplace=True)
df.dropna(axis=1, how="all", inplace=True)

print(f"Removed fully-null rows/cols: {before_shape} -> {df.shape}")

Removed fully-null rows/cols: (505000, 23) -> (505000, 23)


In [5]:
# Cell 5: Strip leading/trailing whitespace and tabs from every object column
obj_cols = df.select_dtypes(include="object").columns.tolist()
print(f"Object columns to clean: {len(obj_cols)}")

for col in obj_cols:
    df[col] = df[col].apply(
        lambda x: re.sub(r"\s+", " ", x).strip() if isinstance(x, str) else x
    )

# Convert empty strings to proper NaN
df[obj_cols] = df[obj_cols].replace({"": np.nan, " ": np.nan})

print("Whitespace stripped and empty strings converted to NaN.")
print("Remaining object columns:", len(obj_cols))

Object columns to clean: 13
Whitespace stripped and empty strings converted to NaN.
Remaining object columns: 13


In [6]:
# Cell 6: Normalize casing for categorical text columns
# Strategy: lower-case first, then apply controlled title-casing to known columns.

text_cols = ["merchant", "merchant_category", "card_type", "country",
             "city", "device_type", "channel", "customer_id",
             "transaction_id", "ip_address"]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Lower-case country ISO codes
if "country" in df.columns:
    df["country"] = df["country"].str.upper()

# Merchant: Title case for readability
if "merchant" in df.columns:
    df["merchant"] = df["merchant"].str.title()

# Merchant category: Title case
if "merchant_category" in df.columns:
    df["merchant_category"] = df["merchant_category"].str.title()

# Card type: Title case then normalize known variants
if "card_type" in df.columns:
    df["card_type"] = df["card_type"].str.title()

# Channel: Title case
if "channel" in df.columns:
    df["channel"] = df["channel"].str.title()

# City: Title case
if "city" in df.columns:
    df["city"] = df["city"].str.title()

# Device: Title case
if "device_type" in df.columns:
    df["device_type"] = df["device_type"].str.title()

print("Text casing normalized.")
print(df[["merchant", "card_type", "channel", "device_type"]].head())

Text casing normalized.
  merchant   card_type channel device_type
0   Zomato  Mastercard  Online         Web
1   Paypal        Visa     Pos      Mobile
2  Bestbuy        Visa     Atm         NaN
3      NaN        Visa     Pos      Tablet
4   Airbnb         NaN     Pos         NaN


In [7]:
# Cell 7: Map sentinel/placeholder strings to NaN
sentinel_values = {
    "n/a", "na", "null", "none", "nan", "unknown", "-", "?", "",
    "not available", "undefined", "missing", "nil"
}

obj_cols = df.select_dtypes(include="object").columns
for col in obj_cols:
    df[col] = df[col].apply(
        lambda x: np.nan if isinstance(x, str) and x.lower() in sentinel_values else x
    )

print("Sentinel values converted to NaN.")
print("Remaining missing cells:", int(df.isna().sum().sum()))

Sentinel values converted to NaN.
Remaining missing cells: 672673


In [8]:
# Cell 8: Canonicalize country names to ISO-2 codes
country_map = {
    "USA": "US", "UNITED STATES": "US", "UNITED STATES OF AMERICA": "US",
    "INDIA": "IN", "BHARAT": "IN",
    "UK": "GB", "UNITED KINGDOM": "GB", "ENGLAND": "GB",
    "GERMANY": "DE", "DEUTSCHLAND": "DE",
    "CANADA": "CA", "FRANCE": "FR", "BRAZIL": "BR", "AUSTRALIA": "AU",
    "RUSSIA": "RU", "NIGERIA": "NG", "CHINA": "CN", "PAKISTAN": "PK",
    "NORTH KOREA": "KP", "IRAN": "IR",
}

if "country" in df.columns:
    df["country"] = df["country"].str.upper()
    df["country"] = df["country"].replace(country_map)
    valid_iso2 = {"US","IN","GB","DE","CA","FR","BR","AU","RU","NG",
                  "CN","PK","KP","IR"}
    df.loc[~df["country"].isin(valid_iso2), "country"] = np.nan

print("Country values standardized. Value counts:")
print(df["country"].value_counts(dropna=False))

Country values standardized. Value counts:
country
US     47567
NaN    46065
RU     31964
BR     31906
GB     31876
NG     31858
FR     31663
IN     31649
PK     31601
CA     31532
DE     31501
AU     31479
IR     31469
KP     31454
CN     31416
Name: count, dtype: int64


In [9]:
# Cell 9: Canonicalize card_type and device_type
card_map = {
    "Visa": "Visa",
    "Mastercard": "MasterCard", "Master Card": "MasterCard",
    "Amex": "AMEX", "American Express": "AMEX",
    "Discover": "Discover",
    "Diners Club": "Diners Club",
}

device_map = {
    "Ios": "iOS", "Iphone": "iOS",
    "Android": "Android", "Samsung": "Android", "Xiomi": "Android",
    "Web": "Web", "Desktop": "Web",
    "Mobile": "Mobile", "Tablet": "Tablet",
}

if "card_type" in df.columns:
    df["card_type"] = df["card_type"].replace(card_map)

if "device_type" in df.columns:
    df["device_type"] = df["device_type"].replace(device_map)

print("Card type values:")
print(df["card_type"].value_counts(dropna=False))
print("\nDevice type values:")
print(df["device_type"].value_counts(dropna=False))

Card type values:
card_type
Visa           147886
MasterCard     147365
AMEX           110478
Discover        37052
Diners Club     36980
NaN             25239
Name: count, dtype: int64

Device type values:
device_type
NaN        133974
iOS        114573
Android    114357
Web         85244
Mobile      28554
Tablet      28298
Name: count, dtype: int64


In [10]:
# Cell 10: Clean 'amount' column - remove currency text, coerce to numeric
def clean_amount(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip().replace(",", "")
    s = re.sub(r"[^\d\.\-]", "", s)
    if s in ("", "-", ".", "-."):
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

df["amount"] = df["amount"].apply(clean_amount)

# Ensure no infinities
df["amount"] = df["amount"].replace([np.inf, -np.inf], np.nan)

print("Amount column cleaned.")
print("Amount stats:")
print(df["amount"].describe())

Amount column cleaned.
Amount stats:
count       479,789.00
mean         27,052.90
std       1,550,412.79
min          -4,805.08
25%              17.35
50%              34.83
75%              71.28
max     100,000,000.00
Name: amount, dtype: float64


In [11]:
# Cell 11: Coerce remaining numeric columns to proper types
numeric_cols = [
    "amount", "avg_amount_30d", "credit_limit", "account_age_days",
    "distance_from_home_km", "txn_count_10min", "txn_count_24h",
    "hour_of_day", "card_last4", "is_international", "is_night",
    "fraud_label",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Numeric dtypes after coercion:")
print(df[numeric_cols].dtypes)

Numeric dtypes after coercion:
amount                   float64
avg_amount_30d           float64
credit_limit             float64
account_age_days         float64
distance_from_home_km    float64
txn_count_10min            int64
txn_count_24h              int64
hour_of_day              float64
card_last4               float64
is_international           int64
is_night                   int64
fraud_label                int64
dtype: object


In [12]:
# Cell 12: Parse timestamp column with mixed formats (datetime, str, epoch int)
def parse_timestamp(x):
    if pd.isna(x):
        return pd.NaT
    if isinstance(x, pd.Timestamp):
        return x
    if isinstance(x, (int, np.integer)):
        try:
            return pd.to_datetime(int(x), unit="s", errors="coerce")
        except Exception:
            return pd.NaT
    s = str(x).strip()
    if s.isdigit():
        try:
            return pd.to_datetime(int(s), unit="s", errors="coerce")
        except Exception:
            return pd.NaT
    return pd.to_datetime(s, errors="coerce")

df["timestamp"] = df["timestamp"].apply(parse_timestamp)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# Drop rows where timestamp cannot be recovered
invalid_ts = df["timestamp"].isna().sum()
print(f"Unparseable timestamps: {invalid_ts:,}")

print("Timestamp column cleaned.")
print(f"Range: {df['timestamp'].min()} to {df['timestamp'].max()}")

Unparseable timestamps: 144
Timestamp column cleaned.
Range: 2023-01-01 00:04:00 to 2024-12-30 23:50:00


In [13]:
# Cell 13: Enforce valid ranges for domain-specific fields
# Hour of day: 0-23
df.loc[~df["hour_of_day"].between(0, 23), "hour_of_day"] = np.nan

# Account age: 0 to 50 years in days (18250 days)
df.loc[df["account_age_days"] < 0, "account_age_days"] = np.nan
df.loc[df["account_age_days"] > 18250, "account_age_days"] = np.nan

# Credit limit: must be positive, capped at 10M
df.loc[df["credit_limit"] <= 0, "credit_limit"] = np.nan
df.loc[df["credit_limit"] > 10_000_000, "credit_limit"] = np.nan

# Distance: must be non-negative, capped at 20000 km (half of Earth's circumference * some buffer)
df.loc[df["distance_from_home_km"] < 0, "distance_from_home_km"] = np.nan
df.loc[df["distance_from_home_km"] > 20000, "distance_from_home_km"] = np.nan

# Transaction counts: non-negative integers
df.loc[df["txn_count_10min"] < 0, "txn_count_10min"] = np.nan
df.loc[df["txn_count_24h"] < 0, "txn_count_24h"] = np.nan

# card_last4: must be 4-digit
def fix_last4(x):
    if pd.isna(x):
        return np.nan
    try:
        v = int(float(x))
        return v if 0 <= v <= 9999 else np.nan
    except Exception:
        return np.nan

df["card_last4"] = df["card_last4"].apply(fix_last4)

# Binary flags: must be 0 or 1
for col in ["is_international", "is_night", "fraud_label"]:
    df.loc[~df[col].isin([0, 1]), col] = np.nan

print("Range validation complete.")

Range validation complete.


In [14]:
# Cell 14: Validate IPv4 addresses, set invalid ones to NaN
ipv4_pattern = re.compile(
    r"^(?:(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}"
    r"(?:25[0-5]|2[0-4]\d|[01]?\d\d?)$"
)

def clean_ip(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    return s if ipv4_pattern.match(s) else np.nan

df["ip_address"] = df["ip_address"].apply(clean_ip)

valid_ips = df["ip_address"].notna().sum()
print(f"Valid IPv4 addresses retained: {valid_ips:,}")
print(f"Invalid/missing IPs set to NaN: {df['ip_address'].isna().sum():,}")

Valid IPv4 addresses retained: 443,144
Invalid/missing IPs set to NaN: 61,856


In [15]:
# Cell 15: Handle duplicate transaction_id values (data-entry errors)
dupe_id_mask = df["transaction_id"].duplicated(keep=False)
dupe_count = dupe_id_mask.sum()

print(f"Rows with duplicated transaction_id: {dupe_count:,}")

# Keep the first occurrence of each transaction_id, mark the rest by appending suffix
if dupe_count > 0:
    df = df.sort_values("timestamp", na_position="last").reset_index(drop=True)
    seen = {}
    new_ids = []
    for tid in df["transaction_id"]:
        if pd.isna(tid):
            new_ids.append(np.nan)
            continue
        if tid in seen:
            seen[tid] += 1
            new_ids.append(f"{tid}-DUP{seen[tid]}")
        else:
            seen[tid] = 0
            new_ids.append(tid)
    df["transaction_id"] = new_ids

print(f"Unique transaction_id after resolution: {df['transaction_id'].nunique():,}")

Rows with duplicated transaction_id: 13,048
Unique transaction_id after resolution: 502,459


In [16]:
# Cell 16: Remove full-row duplicates (business = same txn entered twice)
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)
print(f"Exact duplicate rows removed: {before - after:,}")
print(f"Shape after dedup: {df.shape}")
df.reset_index(drop=True, inplace=True)

Exact duplicate rows removed: 35
Shape after dedup: (504965, 23)


In [17]:
# Cell 17: Impute / drop missing values using column-specific strategy
# Numeric: median (robust to outliers). Categorical: mode or 'Unknown'.
# Rows missing transaction_id or timestamp are unusable -> drop.

critical_cols = ["transaction_id", "timestamp"]
df.dropna(subset=critical_cols, inplace=True)
print(f"Rows dropped for missing critical fields: "
      f"(now {len(df):,} rows)")

numeric_impute = {
    "amount": "median",
    "avg_amount_30d": "median",
    "credit_limit": "median",
    "account_age_days": "median",
    "distance_from_home_km": "median",
    "txn_count_10min": "median",
    "txn_count_24h": "median",
    "hour_of_day": "median",
    "card_last4": "median",
}

for col, strategy in numeric_impute.items():
    if col in df.columns:
        fill_value = df[col].median() if strategy == "median" else df[col].mean()
        df[col] = df[col].fillna(fill_value)

categorical_impute = ["merchant", "merchant_category", "card_type",
                      "country", "city", "device_type", "channel",
                      "ip_address", "customer_id"]

for col in categorical_impute:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

for col in ["is_international", "is_night", "fraud_label"]:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()[0]).astype(int)

print("\nRemaining missing cells:", int(df.isna().sum().sum()))

Rows dropped for missing critical fields: (now 502,316 rows)

Remaining missing cells: 0


In [18]:
# Cell 18: Cap extreme values using IQR-based winsorization
def winsorize_iqr(series, k=3.0):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return series.clip(lower=lower, upper=upper)

continuous_cols = ["amount", "avg_amount_30d", "credit_limit",
                   "distance_from_home_km", "account_age_days",
                   "txn_count_10min", "txn_count_24h"]

for col in continuous_cols:
    if col in df.columns:
        df[col] = winsorize_iqr(df[col], k=3.0)

print("Outliers capped via IQR winsorization (k=3).")
print(df[continuous_cols].describe().T[["min", "max", "mean", "std"]])

Outliers capped via IQR winsorization (k=3).
                          min          max       mean        std
amount                -132.11       218.38      55.04      55.80
avg_amount_30d         -67.35       134.32      38.98      29.87
credit_limit             1.00 1,000,000.00 359,718.98 342,905.47
distance_from_home_km    0.00        95.03      22.50      23.28
account_age_days         1.00     4,999.00   2,385.51   1,464.60
txn_count_10min          0.00         8.00       1.19       1.36
txn_count_24h            0.00        10.00       3.23       2.02


In [19]:
# Cell 19: Cross-field consistency + final audit
# is_night must match hour_of_day
df["is_night_expected"] = df["hour_of_day"].apply(
    lambda h: 1 if 0 <= h < 5 else 0
)
mismatch = (df["is_night"] != df["is_night_expected"]).sum()
df["is_night"] = df["is_night_expected"]
df.drop(columns=["is_night_expected"], inplace=True)
print(f"is_night corrected for {mismatch:,} rows.")

# Derived hour features from timestamp
df["year"] = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["day"] = df["timestamp"].dt.day
df["hour"] = df["timestamp"].dt.hour
df["weekday"] = df["timestamp"].dt.weekday

print("\nFinal consistency audit:")
print(f"Rows            : {len(df):,}")
print(f"Columns         : {len(df.columns)}")
print(f"Missing cells   : {int(df.isna().sum().sum()):,}")
print(f"Duplicate rows  : {int(df.duplicated().sum()):,}")
print(f"Fraud rate      : {df['fraud_label'].mean():.4f}")
print(f"Fraud count     : {int(df['fraud_label'].sum()):,}")
print(f"Date range      : {df['timestamp'].min()} -> {df['timestamp'].max()}")

is_night corrected for 2,268 rows.

Final consistency audit:
Rows            : 502,316
Columns         : 28
Missing cells   : 0
Duplicate rows  : 0
Fraud rate      : 0.0500
Fraud count     : 25,121
Date range      : 2023-01-01 00:04:00 -> 2024-12-30 23:50:00


In [20]:
# Cell 20: Save cleaned dataset and compare to baseline
df.to_csv(CLEAN_PATH, index=False)

post_clean = {
    "rows": len(df),
    "columns": len(df.columns),
    "missing_cells": int(df.isna().sum().sum()),
    "missing_pct": round(df.isna().sum().sum() / df.size * 100, 2),
    "duplicate_rows": int(df.duplicated().sum()),
    "duplicate_txn_id": int(df["transaction_id"].duplicated().sum()),
    "fraud_rate": round(df["fraud_label"].mean(), 4),
}

audit_df = pd.DataFrame({
    "baseline": baseline,
    "post_clean": post_clean,
})
audit_df["delta"] = audit_df["post_clean"] - audit_df["baseline"]

print("CLEANING AUDIT")
print(audit_df)
print(f"\nCleaned dataset saved to: {CLEAN_PATH}")
print(f"File size: {os.path.getsize(CLEAN_PATH) / 1e6:.2f} MB")

print("\nPreview of cleaned data:")
print(df.head())

CLEANING AUDIT
                   baseline  post_clean       delta
rows             505,000.00  502,316.00   -2,684.00
columns               23.00       28.00        5.00
missing_cells    602,339.00        0.00 -602,339.00
missing_pct            5.19        0.00       -5.19
duplicate_rows     4,181.00        0.00   -4,181.00
duplicate_txn_id   7,795.00        0.00   -7,795.00
fraud_rate             0.05        0.05        0.00

Cleaned dataset saved to: /kaggle/working/cleaned_fraud_data.csv
File size: 89.06 MB

Preview of cleaned data:
  transaction_id customer_id           timestamp  amount   merchant merchant_category   card_type  card_last4 country     city device_type      ip_address  is_international  txn_count_10min  \
0   TXN000000001  CUST010205 2023-01-01 00:04:00   26.19    Alibaba       Electronics  MasterCard    5,506.00      NG  Beijing     Unknown         Unknown                 0             3.00   
1   TXN000000000  CUST006550 2023-01-01 00:04:00   52.83    Unknown    

In [32]:
from IPython.display import display
display(df)

,transaction_id,customer_id,timestamp,amount,merchant,merchant_category,card_type,card_last4,country,city,device_type,ip_address,is_international,txn_count_10min,txn_count_24h,avg_amount_30d,account_age_days,credit_limit,hour_of_day,is_night,distance_from_home_km,channel,fraud_label
0,TXN000104241,CUST034785,2023-06-01 23:25:00,1.68,Zomato,Travel,MASTERCARD,"8,711.00",KP,Moscow,web,NaN,1,1,4,35.68,"1,887.00","500,000.00",23.00,0,34.69,Online,0
1,TXN000199676,CUST029737,2023-10-19 21:12:00,36.3,PayPal,Groceries,Visa,"2,558.00",US,Berlin,Mobile,9.132.47.196,0,0,5,9.66,"4,040.00","1,000,000.00",21.00,0,16.23,pos,0
2,TXN000140199,CUST006649,2023-07-24 11:53:00,208.06,BestBuy,Fuel,visa,"7,896.00",DE,Paris,NaN,13.42.180.254,0,0,2,102.92,"4,183.00","1,000,000.00",11.00,0,75.44,ATM,0
3,TXN000132814,CUST022654,2023-07-13 18:11:00,81.1,NaN,Gambling,Visa,"5,692.00",FR,London,Tablet,175.180.134.50,0,1,3,16.72,"3,040.00","1,000,000.00",18.00,0,5.98,pos,0
4,TXN000408697,CUST017646,2024-08-20 01:09:00,12.38,Airbnb,Gift Cards,NaN,"3,965.00",CN,Lagos,NaN,131.152.190.3,0,2,4,446.07,902.00,"1,000,000.00",1.00,1,2.69,POS,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
504995,TXN000353004,CUST006216,2024-05-30 23:04:00,46.51,Delta Air,Luxury,Visa,"2,621.00",PK,Moscow,ios,NaN,0,1,3,9.12,"2,914.00","500,000.00",23.00,0,17.77,Cheque,0
504996,TXN000162926,CUST000826,2023-08-26 19:20:00,12.58,Binance,Wire Transfer,mastercard,"3,115.00",DE,LA,NaN,4.218.74.90,0,4,3,72.39,"2,577.00","500,000.00",19.00,0,2.98,Online,0
504997,TXN000334523,CUST048108,2024-05-03 23:53:00,54.09,Walmart,Food,AMEX,"3,015.00",CA,Delhi,android,NaN,0,0,7,11.99,"2,322.00","1,000,000.00",23.00,0,7.51,App,0
504998,TXN000363030,CUST048393,2024-06-14 14:59:00,208.63,Starbucks,Adult,Visa,"9,800.00",CN,NYC,Desktop,68.178.192.100,0,1,2,8.03,"1,302.00","200,000.00",14.00,0,15.96,Contactless,0


In [34]:
import os
print("File exists:", os.path.exists("/kaggle/working/cleaned_fraud_data.csv"))
print("Files in /kaggle/working:", os.listdir("/kaggle/working"))

File exists: True
Files in /kaggle/working: ['.virtual_documents', 'cleaned_fraud_data.csv']


# Analysis 